In [1]:
import pandas as pd

df = pd.read_excel("/content/Resturent_items.xlsx")

print("Dataset loaded successfully!")
print("Number of rows:", len(df))
print("Number of columns:", len(df.columns))

display(df.head())

Dataset loaded successfully!
Number of rows: 100
Number of columns: 8


,Order_ID,Customer_Name,Food_Item,Quantity,Order_Type,Preparation_Time_Min,Delivery_Time_Min,Order_Status
0,1,Customer_1,Salad,4,Delivery,18,16,Delivered
1,2,Customer_2,Pasta,3,Dine-in,22,0,Delivered
2,3,Customer_3,Noodles,4,Delivery,15,13,Cancelled
3,4,Customer_4,Pasta,4,Takeaway,15,0,Delivered
4,5,Customer_5,Pizza,2,Delivery,9,37,Delivered


In [4]:
!pip install -q opentelemetry-api==1.42.1

In [5]:
import prefect
print("Prefect installed successfully!")
print("Prefect version:", prefect.__version__)

Prefect installed successfully!
Prefect version: 3.9.1


In [6]:
from prefect import task, flow

@task
def collect_order():
    print("Customer order collected")
    return "Pizza"

@task
def prepare_order(order):
    print(f"Preparing {order}")
    return f"{order} is ready"

@task
def deliver_order(prepared_order):
    print(f"Delivering order: {prepared_order}")
    return "Order delivered successfully"

@flow
def restaurant_workflow():
    order = collect_order()
    prepared_order = prepare_order(order)
    result = deliver_order(prepared_order)
    print(result)

restaurant_workflow()

INFO:prefect:Starting temporary server on http://127.0.0.1:8854
See https://www.prefect.io/docs/v3/concepts/server#how-to-guides for more information on running a dedicated Prefect server.
INFO:prefect.flow_runs:Beginning flow run 'coral-bison' for flow 'restaurant-workflow'
INFO:prefect.task_runs:Finished in state Completed()
INFO:prefect.task_runs:Finished in state Completed()
INFO:prefect.task_runs:Finished in state Completed()


Customer order collected
Preparing Pizza
Delivering order: Pizza is ready
Order delivered successfully


INFO:prefect.flow_runs:Finished in state Completed()


In [7]:
from prefect import task, flow
import pandas as pd

@task
def load_data():
    df = pd.read_excel("/content/Resturent_items.xlsx")
    print("Restaurant dataset loaded successfully!")
    print("Dataset shape:", df.shape)
    return df

@flow
def restaurant_data_flow():
    data = load_data()
    print("\nFirst 5 records:")
    print(data.head())

restaurant_data_flow()

INFO:prefect.flow_runs:Beginning flow run 'bouncy-ibex' for flow 'restaurant-data-flow'
INFO:prefect.task_runs:Finished in state Completed()


Restaurant dataset loaded successfully!
Dataset shape: (100, 8)

First 5 records:
   Order_ID Customer_Name Food_Item  Quantity Order_Type  \
0         1    Customer_1     Salad         4   Delivery   
1         2    Customer_2     Pasta         3    Dine-in   
2         3    Customer_3   Noodles         4   Delivery   
3         4    Customer_4     Pasta         4   Takeaway   
4         5    Customer_5     Pizza         2   Delivery   

   Preparation_Time_Min  Delivery_Time_Min Order_Status  
0                    18                 16    Delivered  
1                    22                  0    Delivered  
2                    15                 13    Cancelled  
3                    15                  0    Delivered  
4                     9                 37    Delivered  


INFO:prefect.flow_runs:Finished in state Completed()


In [8]:
from prefect import task, flow
import pandas as pd

@task
def load_data():
    return pd.read_excel("/content/Resturent_items.xlsx")

@task
def preprocess_data(df):
    df = df.copy()

    # Remove customer name
    df = df.drop(columns=["Customer_Name"])

    # Convert Order_Type into numerical values
    df["Order_Type"] = df["Order_Type"].map({
        "Delivery": 1,
        "Takeaway": 2,
        "Dine-in": 3
    })

    # Convert Order_Status into numerical values
    df["Order_Status"] = df["Order_Status"].map({
        "Cancelled": 0,
        "Delivered": 1,
        "Preparing": 2
    })

    # Convert Food_Item into numerical category codes
    df["Food_Item"] = df["Food_Item"].astype("category").cat.codes

    print("Data preprocessing completed!")
    print("Processed dataset shape:", df.shape)

    return df

@flow
def restaurant_preprocessing_flow():
    data = load_data()
    processed_data = preprocess_data(data)

    print("\nProcessed dataset:")
    print(processed_data.head())

restaurant_preprocessing_flow()

INFO:prefect.flow_runs:Beginning flow run 'romantic-owl' for flow 'restaurant-preprocessing-flow'
INFO:prefect.task_runs:Finished in state Completed()
INFO:prefect.task_runs:Finished in state Completed()


Data preprocessing completed!
Processed dataset shape: (100, 7)

Processed dataset:
   Order_ID  Food_Item  Quantity  Order_Type  Preparation_Time_Min  \
0         1          6         4           1                    18   
1         2          4         3           3                    22   
2         3          3         4           1                    15   
3         4          4         4           2                    15   
4         5          5         2           1                     9   

   Delivery_Time_Min  Order_Status  
0                 16             1  
1                  0             1  
2                 13             0  
3                  0             1  
4                 37             1  


INFO:prefect.flow_runs:Finished in state Completed()


In [9]:
from prefect import task, flow
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

@task
def train_model(df):
    X = df.drop(columns=["Order_Status", "Order_ID"])
    y = df["Order_Status"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    model = RandomForestClassifier(random_state=42)
    model.fit(X_train, y_train)

    print("Model training completed successfully!")

    return model, X_test, y_test

@flow
def restaurant_training_flow():
    data = pd.read_excel("/content/Resturent_items.xlsx")

    data = data.drop(columns=["Customer_Name"])
    data["Order_Type"] = data["Order_Type"].map({
        "Delivery": 1, "Takeaway": 2, "Dine-in": 3
    })
    data["Order_Status"] = data["Order_Status"].map({
        "Cancelled": 0, "Delivered": 1, "Preparing": 2
    })
    data["Food_Item"] = data["Food_Item"].astype("category").cat.codes

    model, X_test, y_test = train_model(data)

    predictions = model.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)

    print("Model Accuracy:", round(accuracy * 100, 2), "%")

restaurant_training_flow()

INFO:prefect.flow_runs:Beginning flow run 'defiant-ostrich' for flow 'restaurant-training-flow'
INFO:prefect.task_runs:Finished in state Completed()


Model training completed successfully!
Model Accuracy: 75.0 %


INFO:prefect.flow_runs:Finished in state Completed()


In [10]:
from prefect import task, flow
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

@task
def evaluate_model(model, X_test, y_test):
    predictions = model.predict(X_test)

    print("Model Evaluation Results")
    print("------------------------")
    print("Accuracy:", round(accuracy_score(y_test, predictions), 4))
    print("Precision:", round(
        precision_score(y_test, predictions, average="weighted", zero_division=0), 4
    ))
    print("Recall:", round(
        recall_score(y_test, predictions, average="weighted", zero_division=0), 4
    ))
    print("F1 Score:", round(
        f1_score(y_test, predictions, average="weighted", zero_division=0), 4
    ))

    print("\nClassification Report:")
    print(classification_report(y_test, predictions, zero_division=0))

@flow
def restaurant_evaluation_flow():
    data = pd.read_excel("/content/Resturent_items.xlsx")

    data = data.drop(columns=["Customer_Name"])
    data["Order_Type"] = data["Order_Type"].map({
        "Delivery": 1, "Takeaway": 2, "Dine-in": 3
    })
    data["Order_Status"] = data["Order_Status"].map({
        "Cancelled": 0, "Delivered": 1, "Preparing": 2
    })
    data["Food_Item"] = data["Food_Item"].astype("category").cat.codes

    X = data.drop(columns=["Order_Status", "Order_ID"])
    y = data["Order_Status"]

    from sklearn.model_selection import train_test_split
    from sklearn.ensemble import RandomForestClassifier

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    model = RandomForestClassifier(random_state=42)
    model.fit(X_train, y_train)

    evaluate_model(model, X_test, y_test)

restaurant_evaluation_flow()

INFO:prefect.flow_runs:Beginning flow run 'didactic-kangaroo' for flow 'restaurant-evaluation-flow'
INFO:prefect.task_runs:Finished in state Completed()


Model Evaluation Results
------------------------
Accuracy: 0.75
Precision: 0.7353
Recall: 0.75
F1 Score: 0.7204

Classification Report:
              precision    recall  f1-score   support

           0       0.67      0.33      0.44         6
           1       0.76      0.93      0.84        14

    accuracy                           0.75        20
   macro avg       0.72      0.63      0.64        20
weighted avg       0.74      0.75      0.72        20



INFO:prefect.flow_runs:Finished in state Completed()


In [11]:
from prefect import task, flow

attempt_count = 0

@task(retries=2, retry_delay_seconds=2)
def connect_restaurant_database():
    global attempt_count
    attempt_count += 1

    print("Connecting to restaurant database...")

    if attempt_count == 1:
        raise Exception("Temporary database connection failure")

    print("Database connected successfully!")
    return "Connection successful"

@task
def preprocess_restaurant_data(connection):
    print("Preprocessing restaurant data...")
    return "Restaurant data preprocessing completed"

@flow
def restaurant_retry_flow():
    connection = connect_restaurant_database()
    result = preprocess_restaurant_data(connection)
    print(result)

restaurant_retry_flow()

INFO:prefect.flow_runs:Beginning flow run 'imposing-serval' for flow 'restaurant-retry-flow'
INFO:prefect.task_runs:Task run failed with exception: Exception('Temporary database connection failure') - Retry 1/2 will start 2 second(s) from now


Connecting to restaurant database...


INFO:prefect.task_runs:Finished in state Completed()
INFO:prefect.task_runs:Finished in state Completed()


Connecting to restaurant database...
Database connected successfully!
Preprocessing restaurant data...
Restaurant data preprocessing completed


INFO:prefect.flow_runs:Finished in state Completed()


In [12]:
from prefect import task, flow

@task
def train_acceptance_model():
    print("Training model...")
    accuracy = 0.87
    print("Model training completed!")
    return accuracy

@task
def evaluate_acceptance_model(accuracy):
    print("Evaluating model...")
    print("Model Accuracy:", accuracy)
    return accuracy >= 0.85

@task
def save_acceptance_model():
    print("Model accepted!")
    print("Model saved successfully!")

@flow
def restaurant_model_acceptance_flow():
    accuracy = train_acceptance_model()
    accepted = evaluate_acceptance_model(accuracy)

    if accepted:
        save_acceptance_model()
    else:
        print("Model rejected! Accuracy is below 85%.")

restaurant_model_acceptance_flow()

INFO:prefect.flow_runs:Beginning flow run 'chestnut-wasp' for flow 'restaurant-model-acceptance-flow'
INFO:prefect.task_runs:Finished in state Completed()
INFO:prefect.task_runs:Finished in state Completed()
INFO:prefect.task_runs:Finished in state Completed()


Training model...
Model training completed!
Evaluating model...
Model Accuracy: 0.87
Model accepted!
Model saved successfully!


INFO:prefect.flow_runs:Finished in state Completed()


In [13]:
import joblib
from prefect import task, flow
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

@task
def train_and_save_model():
    data = pd.read_excel("/content/Resturent_items.xlsx")

    data = data.drop(columns=["Customer_Name"])

    data["Order_Type"] = data["Order_Type"].map({
        "Delivery": 1,
        "Takeaway": 2,
        "Dine-in": 3
    })

    data["Order_Status"] = data["Order_Status"].map({
        "Cancelled": 0,
        "Delivered": 1,
        "Preparing": 2
    })

    data["Food_Item"] = data["Food_Item"].astype("category").cat.codes

    X = data.drop(columns=["Order_Status", "Order_ID"])
    y = data["Order_Status"]

    model = RandomForestClassifier(random_state=42)
    model.fit(X, y)

    joblib.dump(model, "/content/restaurant_model.pkl")

    print("Random Forest model trained successfully!")
    print("Model saved as restaurant_model.pkl")

@flow
def save_restaurant_model_flow():
    train_and_save_model()

save_restaurant_model_flow()

INFO:prefect.flow_runs:Beginning flow run 'glistening-ant' for flow 'save-restaurant-model-flow'
INFO:prefect.task_runs:Finished in state Completed()


Random Forest model trained successfully!
Model saved as restaurant_model.pkl


INFO:prefect.flow_runs:Finished in state Completed()


In [14]:
from google.colab import files

files.download("/content/restaurant_model.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>